# M24 · Counterfactual / off-policy evaluation

_Curriculum · Domain 5 · Reinforcement learning_

**Estimate a new ads policy from old logged traffic before risking an online test.**

We compute IPS, self-normalized IPS, and Doubly Robust estimates on a tiny logged bandit table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(24)

## Logged bandit feedback

Each row contains context, logged action, logging propensity $p_{log}(a|x)$, and observed reward $r$. Off-policy evaluation asks for the value of a new policy $\pi_{new}$ using only those rows.

In [ ]:
log = pd.DataFrame({
    "segment": ["cold", "cold", "warm", "warm", "hot", "hot"],
    "action": ["A", "B", "A", "C", "B", "C"],
    "p_log": [0.50, 0.25, 0.40, 0.20, 0.30, 0.35],
    "reward": [0.0, 1.0, 0.0, 1.0, 1.0, 0.0],
    "p_new": [0.20, 0.50, 0.10, 0.60, 0.45, 0.25],
})

log["weight"] = log["p_new"] / log["p_log"]
print(log)
assert np.all(log["p_log"] > 0.0)

## IPS

The inverse propensity score estimator reweights rewards by how much more often the new policy would have chosen the logged action:

$$\hat V_{IPS}=\frac{1}{n}\sum_i r_i\frac{\pi_{new}(a_i|x_i)}{p_{log}(a_i|x_i)}.$$

In [ ]:
ips_terms = log["reward"] * log["weight"]
ips_value = ips_terms.mean()

print(ips_terms.round(3).to_list())
print(round(ips_value, 4))
assert np.isclose(ips_value, 1.0833333333333333)

## Self-normalized IPS

SNIPS divides by the total weight. It often lowers variance, but it introduces a little bias in finite samples.

In [ ]:
snips_value = ips_terms.sum() / log["weight"].sum()

print(round(snips_value, 4))
assert 0.80 < snips_value < 0.85

## Doubly Robust

DR uses a reward model $\hat q(x,a)$ plus an IPS correction for the action actually observed. If either the model or propensities are right, DR can be reliable.

In [ ]:
q_new = np.array([0.35, 0.55, 0.40, 0.65, 0.70, 0.50])
q_logged = np.array([0.30, 0.45, 0.35, 0.55, 0.60, 0.45])
correction = log["weight"].to_numpy() * (log["reward"].to_numpy() - q_logged)
dr_terms = q_new + correction
dr_value = dr_terms.mean()

print(np.round(dr_terms, 3))
print(round(dr_value, 4))
assert 0.90 < dr_value < 1.00

## Variance is the warning light

Large importance weights mean a small number of logged rows dominate the estimate. In production we inspect max weight and effective sample size before trusting the number.

In [ ]:
weights = log["weight"].to_numpy()
effective_n = weights.sum() ** 2 / np.sum(weights ** 2)

print("max weight", round(weights.max(), 3))
print("effective n", round(effective_n, 3))
assert effective_n < len(weights)

## Visual check

The estimators answer the same question with different bias-variance trade-offs.

In [ ]:
names = ["IPS", "SNIPS", "DR"]
values = [ips_value, snips_value, dr_value]

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(names, values, color="#f58518")
ax.set_ylim(0.0, 0.9)
ax.set_ylabel("estimated value")
ax.set_title("off-policy estimates")
plt.show()